# Capítulo 5: Formatando Saída e Falando por Claude

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Playground de Exemplos](#example-playground)

## Configuração

Execute a célula de configuração a seguir para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
%pip install anthropic --quiet

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Import python's built-in regular expression library
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ],
        system=system
    )
    return message.content[0].text

---

## Lição

Se você quiser que Claude forneça sua saída de uma certa maneira, **basta pedir!** As formas de orientar a saída de Claude incluem, mas não estão limitadas a:
- Solicitações de formatação de saída em um determinado formato (XML, JSON, etc.)
- Solicitar que Claude forneça diretamente a resposta desejada sem preâmbulo
- Fazer Claude começar sua resposta por você com algumas palavras que você coloca na boca de Claude

Todas as três técnicas acima podem tornar Claude mais preciso, mais consistente e também mais amigável para máquinas (ou seja, saída que pode ser facilmente analisada por código).

Abaixo, demonstraremos cada uma delas.

### Exemplos

Vamos começar solicitando formatação de saída específica de Claude. Aqui está um exemplo sem formatação solicitada.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Agora, aqui está o mesmo prompt, exceto que especificamos que queremos a saída em JSON. Note que Claude adiciona quebras de linha e espaços em branco para facilitar a leitura. Se você gostaria de JSON de uma linha, basta solicitar especificamente isso.

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Também podemos solicitar que Claude forneça a resposta desejada sem nenhum outro texto ao redor dela. Vamos dizer que temos uma tarefa onde queremos extrair o nome do autor de uma sentença. Nosso primeiro prompt pode ser assim:

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

A resposta acima não é ideal se quisermos analisar essa saída e colocá-la em outro lugar. Há muitas palavras adicionais além do nome do autor. Podemos corrigi-lo sendo explícitos sobre o que queremos.

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Por último, podemos fazer Claude começar sua resposta por nós **preenchendo a mensagem `Assistant`**. Isso é particularmente útil quando você quer que Claude responda apenas de uma certa forma ou não quer dar a Claude a oportunidade de se recusar.

Abaixo, usamos esta técnica para fazer Claude responder com apenas JSON, sem nenhum outro texto ao redor.

---

## Exercises
- [Exercise 5.1 - Steph Curry GOAT](#exercise-51---steph-curry-goat)
- [Exercise 5.2 - Two Haikus](#exercise-52---two-haikus)
- [Exercise 5.3 - Two Haikus, Two Animals](#exercise-53---two-haikus-two-animals)

Você pode até colocar algumas tags XML na boca de Claude. O snippet de resposta abaixo faz com que Claude continue a partir de onde você parou, preenchendo o conteúdo da tag `<answer>`. Note que fazemos isso adicionando o snippet de resposta ao final das mensagens de prompt como um turno `Assistant`, mas sem incluir o conteúdo `</answer>`.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Prefill for Claude's response
PREFILL = ""

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

**Observações importantes ao prefixar a resposta de Claude:**
1. Quando você usa este método, lembre-se de que **o texto que você colocou na boca de Claude será contado e cobrado como uma saída gerada por Claude**, mesmo que tecnicamente você o tenha escrito. Por exemplo, na chamada da API acima, embora Claude tenha realmente produzido apenas o texto "Sim", a saída completa do turno `Assistant` será o que quer que você o tenha preenchido, neste caso "\n\n<resposta>Sim". Isso significa que você será cobrado pelos tokens nesse texto preenchido também.
2. Quando você usa este método, **Claude pode ocasionalmente se descarrilar ou ignorar sua tentativa de prefixar sua resposta**, especialmente se você não incluir exemplos ou instruções adicionais.

Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar o [**Playground de Exemplos**](#example-playground).

In [ ]:
print(hints.exercise_5_1_hint)

### Exercício 5.1 - Formatação JSON
Modifique o `PROMPT` para tornar Claude produzir a resposta no formato JSON.

In [ ]:
# Variable content
ANIMAL = "cats"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
print(hints.exercise_5_2_hint)

### Exercício 5.2 - Começando a Resposta de Claude

Modifique o turno `Assistant` preenchido (a resposta que começa para Claude) para fazer Claude produzir **APENAS** o nome do modelo de idioma grande.

In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
print(hints.exercise_5_3_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Bons prompts!

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente com os exemplos de prompts mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas de Claude.

In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))